# Compare PR #702 `_r.dat` vs postw90 `_r_full.dat`

Checks that Wannier90 PR #702's `_r.dat` is postw90's `A(R)`, rounded to `6F12.6`.

- **postw90 `_r_full.dat`**: `get_AA_R` with `transl_inv_full=T`, exported by the local `write_aa_r` patch. Written at `ES24.16E3`.
- **wannier90.x `_r.dat`**: `plot_write_rmn` with `transl_inv_full=T`, added by upstream PR #702. Written at `6F12.6`.

If the two are the same matrix, every component of `_r.dat` must be *exactly* the six-decimal rendering of the corresponding `_r_full.dat` value -- bit-exact after rounding, not "agrees to tolerance". Also reports what the `6F12.6` format costs on its own, since PR #702 does not change it -- see `notes/progress/2026-08-27_rfull_vs_mmn_position_matrix.md` section 0.

Set `RUN_COMPUTATION = False` to skip straight to loading the last saved run from `results/pr702_rdat_file/`.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

REPO_ROOT = Path.cwd()
while not (REPO_ROOT / "validation").exists():
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT))

from validation._paths import RESULTS
from validation.inputs.compare_pr702_rdat_lib import (
    DEFAULT_CANDIDATE,
    DEFAULT_REFERENCE,
    read_matrix,
    run_case,
    save_outputs,
    plot,
)

## Config

`CASES` is `(label, candidate_path, reference_path, control_path_or_None)`. Add more tuples to compare several systems in one run; `control_path` is the stock (non-`transl_inv_full`) `_r.dat` from the same checkpoint, scored against the same reference.

In [ ]:
RUN_COMPUTATION = False  # True: rebuild and recompute. False: load results/pr702_rdat_file/.
OUTPUT_DIR = RESULTS / "pr702_rdat_file"
CASES = [
    ("SrTiO3 Ti_Q_2A trial-03", DEFAULT_CANDIDATE, DEFAULT_REFERENCE, None),
]

## Run each case

`cases[i]["rows"][j]["result"]` holds the per-file comparison dict (bit-exact fraction, max residual, Hermiticity, ...); `cases[i]["rows"][j]["diff"]` is the raw per-component residual array.

In [ ]:
if RUN_COMPUTATION:
    cases = [run_case(*spec) for spec in CASES]

## Save tables + figure

In [ ]:
if RUN_COMPUTATION:
    for path in save_outputs(OUTPUT_DIR, cases):
        print(f"saved {path}")

## Load a previous run (skip if you just computed above)

In [ ]:
if not RUN_COMPUTATION:
    import csv
    with (OUTPUT_DIR / "file_level_summary.csv").open() as f:
        summary_rows = list(csv.DictReader(f))
    for row in summary_rows:
        print(row["system"], "|", row["file"], "|", row["bit-exact vs 6F12.6(ref)"] if "bit-exact vs 6F12.6(ref)" in row else row.get("bit_exact_fraction"))
    # Full arrays weren't cached to disk (only the summary table + PNG were); rerun with
    # RUN_COMPUTATION = True to get `cases` back as live objects.

## Plot (only available after RUN_COMPUTATION = True in this session)

In [ ]:
if RUN_COMPUTATION:
    fig = plot(cases)
    plt.show()